In [13]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,ImagePath,Label
0,250,dataset/images/sample_0250.png,3
1,434,dataset/images/sample_0434.png,4
2,20,dataset/images/sample_0020.png,4
3,323,dataset/images/sample_0323.png,4
4,333,dataset/images/sample_0333.png,3


In [14]:
df_train.Label.value_counts()

Label
4    101
3     96
5     87
6     75
2     25
7     13
8      3
Name: count, dtype: int64

In [15]:
import torch

num_epochs = 25
batch_size = 32
learning_rate = 0.0004

device = torch.device("cuda")
device

device(type='cuda')

In [16]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
])

valid_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor()
])

class MyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.ImagePath.iloc[idx]
        img = Image.open(path)
        img = self.transform(img)
        label = torch.tensor(self.df.Label.iloc[idx], dtype=torch.float32).view(1)
        return img, label

from sklearn.model_selection import train_test_split
df_1, df_2 = train_test_split(df_train, test_size=0.2, random_state=42)

train_dataset = MyDataset(df_1, train_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size)

valid_dataset = MyDataset(df_2, valid_transform)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)

train_dataset[0][0][0][0][0]

tensor(0.)

In [17]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, X):
        X = self.net(X)
        return X

model = CNN().to(device)
criterion = nn.MSELoss()
optim = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model

CNN(
  (net): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Flatten(start_dim=1, end_dim=-1)
    (10): LazyLinear(in_features=0, out_features=64, bias=True)
    (11): ReLU()
    (12): Linear(in_features=64, out_features=32, bias=True)
    (13): ReLU()
    (14): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [18]:
def train(model, epoch):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optim.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optim.step()
        total_loss += loss.item()
    print("Epoch: ", epoch)
    print("Training loss: ", total_loss/len(inputs))

def test(model, epoch):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    print("Validation loss: ", total_loss/len(inputs))

In [19]:
for epoch in range(1, num_epochs+1):
    train(model, epoch)
    test(model, epoch)

Epoch:  1
Training loss:  3.58529732003808
Validation loss:  0.5841034799814224
Epoch:  2
Training loss:  0.8782192580401897
Validation loss:  0.32575008273124695
Epoch:  3
Training loss:  0.6014490984380245
Validation loss:  0.2930728644132614
Epoch:  4
Training loss:  0.5262703523039818
Validation loss:  0.2485463246703148
Epoch:  5
Training loss:  0.5085168965160847
Validation loss:  0.2600822448730469
Epoch:  6
Training loss:  0.48758935555815697
Validation loss:  0.23461055010557175
Epoch:  7
Training loss:  0.45283109694719315
Validation loss:  0.21560361236333847
Epoch:  8
Training loss:  0.42935460060834885
Validation loss:  0.20418381690979004
Epoch:  9
Training loss:  0.41525618359446526
Validation loss:  0.1981653980910778
Epoch:  10
Training loss:  0.4016404524445534
Validation loss:  0.18977374210953712
Epoch:  11
Training loss:  0.38660685159265995
Validation loss:  0.18466203659772873
Epoch:  12
Training loss:  0.3778884746134281
Validation loss:  0.17813880369067192
Epo

In [20]:
final_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor()
])

class FinalDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.ImagePath.iloc[idx]
        img = Image.open(path)
        img = self.transform(img)
        return img

final_dataset = FinalDataset(df_test, final_transform)
final_loader = DataLoader(final_dataset, batch_size=batch_size)

final_dataset[0][0][0][0]

tensor(0.3882)

In [21]:
import numpy as np
model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to(device)
        outputs = model(inputs).squeeze(1).cpu().numpy()
        outputs = np.round(outputs)
        outputs = np.clip(outputs, 2, 8)
        preds.extend(outputs)

preds[:10]

[4.0, 3.0, 4.0, 4.0, 4.0, 3.0, 4.0, 5.0, 5.0, 3.0]

In [22]:
df_preds = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "PredictedLabel": preds
})

df_preds

,SampleID,PredictedLabel
0,362,4.0
1,74,3.0
2,375,4.0
3,156,4.0
4,105,4.0
...,...,...
95,348,4.0
96,87,4.0
97,76,3.0
98,439,4.0


In [23]:
df_preds.PredictedLabel.value_counts()

PredictedLabel
4.0    42
3.0    27
5.0    27
6.0     4
Name: count, dtype: int64

In [24]:
df_preds.to_csv("submission.csv", index=False)